In [26]:
import pandas as pd
import numpy as np
import math
from scipy.stats import *

In [ ]:
df_5743 = pd.read_csv("gr_linux_43.csv")
df_5746 = pd.read_csv("gr_linux_46.csv")
df_5753 = pd.read_csv("gr_linux_53.csv")
df_js_win = pd.read_csv("js_windows.csv")

In [44]:
test_5743 = df_5743["Duration(ms)"].copy()
test_5746 = df_5746["Duration(ms)"].copy()
test_5753 = df_5753["Duration(ms)"].copy()
test_js_win = df_js_win["Duration(ms)"].copy()

In [ ]:
test_js_win.to_csv("js_windows.csv")

In [45]:
test_js_win.describe()

count       303.000000
mean      14641.782178
std       15959.931143
min         639.000000
25%        5969.500000
50%        8972.000000
75%       18078.500000
max      120756.000000
Name: Duration(ms), dtype: float64

In [48]:
def sums_desctibe(sample, buckets, units='min'):
    rate = 60_000
    if units == 'ms':
        rate = 1

    sample_mean = sample.mean()
    sample_std = sample.std()
    n = sample.count()

    # bucket size
    m = n / buckets

    bucket_mean = sample_mean * n / buckets

    # finite population factor
    C = math.sqrt(m * (n - m) / (n - 1))
    bucket_std = C * sample.std()

    print(f"Estimated mean of bucket sums ({units}): {bucket_mean / rate:.3f}")
    print(f"Estimated std of bucket sums ({units}): {bucket_std / rate:.3f}")

    
    alpha = 0.01
    df = n - 1

    # chi-square quantiles
    chi2_low = chi2.ppf(alpha / 2, df)
    chi2_high = chi2.ppf(1 - alpha / 2, df)

    # confidence interval for original std
    sigma_low = math.sqrt((df * sample_std**2) / chi2_high)
    sigma_high = math.sqrt((df * sample_std**2) / chi2_low)

    # confidence interval for bucket sum std
    bucket_std_low = C * sigma_low
    bucket_std_high = C * sigma_high

    print("\n99% confidence interval:")
    print(f"{bucket_std_low / rate:.3f}, {bucket_std_high / rate:.3f}")

    


In [49]:
sums_desctibe(test_js_win, 4)

Estimated mean of bucket sums (min): 18.485
Estimated std of bucket sums (min): 2.008

99% confidence interval:
1.817, 2.241


In [51]:
def estimate_bucket_risk_two_tailed(values, buckets=12, 
                         threshold=0.15,
                         simulations=10000):

    values = np.asarray(values)

    n = len(values)

    # expected bucket sum
    expected = values.sum() / buckets

    bad_count = 0
    total_buckets = 0

    # store optional results
    max_deviations = []

    for _ in range(simulations):

        shuffled = np.random.permutation(values)

        # split into buckets
        groups = np.array_split(shuffled, buckets)

        sums = np.array([g.sum() for g in groups])

        deviations = np.abs(sums - expected) / expected

        bad_count += np.sum(deviations > threshold)
        total_buckets += buckets

        max_deviations.append(deviations.max())

    probability = bad_count / total_buckets

    return probability, np.array(max_deviations)


In [52]:
values = test_5753.values

p, max_dev = estimate_bucket_risk_two_tailed(values)

print("Probability a bucket exceeds ±15%:", p)
print("Probability at least one bucket exceeds ±15%:",
      np.mean(max_dev > 0.15))

Probability a bucket exceeds ±15%: 0.448125
Probability at least one bucket exceeds ±15%: 1.0


In [53]:
def estimate_bucket_risk(values, 
                         buckets=12, 
                         threshold=0.20, 
                         simulations=100000):

    values = np.asarray(values)

    # expected sum of one bucket
    expected = values.sum() / buckets

    bad_bucket_count = 0
    max_excess = []

    for _ in range(simulations):

        shuffled = np.random.permutation(values)
        groups = np.array_split(shuffled, buckets)
        sums = np.array([g.sum() for g in groups])

        # relative excess of each bucket:
        excess = sums / expected - 1

        # worst bucket from this simulation
        max_excess.append(np.max(excess))

        # count how many bucket allocations exceed the threshold
        bad_bucket_count += np.sum(excess > threshold)
        

    max_excess = np.array(max_excess)

    # probability a random bucket exceeds threshold
    total_bucket_count = simulations * buckets
    p_per_bucket = bad_bucket_count / total_bucket_count

    # probability at least one bucket exceeds threshold
    p_any_bucket = np.mean(max_excess > threshold)

    # statistics of the worst bucket
    mean_max_excess = np.mean(max_excess)
    median_max_excess = np.median(max_excess)

    p95_max_excess = np.percentile(max_excess, 95)
    p99_max_excess = np.percentile(max_excess, 99)

    # convert excess percentages back to sums, and convert to minutes from ms
    rate = 60_000
    max_sum_mean = expected * (1 + mean_max_excess) / rate
    max_sum_median = expected * (1 + median_max_excess) / rate
    max_sum_p95 = expected * (1 + p95_max_excess) / rate
    max_sum_p99 = expected * (1 + p99_max_excess) / rate

    return {
        "Expected bucket sum (min)": 
            expected / rate,

        "Probability that a bucket exceeds the threshold": 
            p_per_bucket,

        "Probability that any bucket in a simulation exceeds the threshold":
            p_any_bucket,

        "Mean maximum excess":
            mean_max_excess,

        "Median maximum excess":
            median_max_excess,

        "95th percentile of maximum excess":
            p95_max_excess,

        "99th percentile of maximum excess":
            p99_max_excess,

        "Mean maximum bucket sum (min)":
            max_sum_mean,
         
        "Median maximum bucket sum (min)":
            max_sum_median,
                
        "95th percentile of maximum bucket sum (min)":
            max_sum_p95,

        "99th percentile of maximum bucket sum (min)":
            max_sum_p99
    }


In [ ]:
values = test_5743.values

result = estimate_bucket_risk(
    values,
    buckets=12,
    threshold=0.20,
    simulations=100000
)

# convert ms to min 
rate = 60_000

for k, v in result.items():
    if k != "max_excess_samples":
        print(f"{k}: {v:.3f}")

Expected bucket sum (min): 13.168
Probability that a bucket exceeds the threshold: 0.121
Probability that any bucket in a simulation exceeds the threshold: 1.000
Mean maximum excess: 0.818
Median maximum excess: 0.810
95th percentile of maximum excess: 1.075
99th percentile of maximum excess: 1.197
Mean maximum bucket sum (min): 23.947
Median maximum bucket sum (min): 23.829
95th percentile of maximum bucket sum (min): 27.331
99th percentile of maximum bucket sum (min): 28.935


In [62]:
values = test_js_win.values

result = estimate_bucket_risk(
    values,
    buckets=3,
    threshold=0.20,
    simulations=100000
)

# convert ms to min 
rate = 60_000

for k, v in result.items():
    if k != "max_excess_samples":
        print(f"{k}: {v:.3f}")

Expected bucket sum (min): 24.647
Probability that a bucket exceeds the threshold: 0.014
Probability that any bucket in a simulation exceeds the threshold: 0.043
Mean maximum excess: 0.094
Median maximum excess: 0.085
95th percentile of maximum excess: 0.194
99th percentile of maximum excess: 0.248
Mean maximum bucket sum (min): 26.958
Median maximum bucket sum (min): 26.752
95th percentile of maximum bucket sum (min): 29.426
99th percentile of maximum bucket sum (min): 30.752
